In [2]:
import pandas as pd

In [3]:
df = pd.read_parquet("hf://datasets/sharjeelyunus/github-issues-dataset/github_issues_dataset.parquet")

c:\Users\arsal\Desktop\support-ticket-triage-system\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df.drop(["id"], axis=1, inplace=True)

In [5]:
df.sample(5)

,repo,title,body,labels,priority,severity
107521,vscode,"With the `Light+` theme, the input area of Ch...",Version: 1.96.0-insider\nCommit: 1bdee4500fc32...,"bug,themes,confirmed,chat",low,Minor
2106,opencv,LineIterator is missing from Java bindings,Title says it all. I now have to implement my ...,"feature,priority: low,category: java bindings",low,Minor
77473,stable-diffusion-webui,[Bug]: HIRES.FIX bad memory management [SOLVED],### Is there an existing issue for this?\n\n- ...,bug-report,low,Critical
74717,vscode,Highlight matched words in similar commands,Testing #194021\n\nIf would be nice if there's...,"feature-request,quick-open",low,Minor
79504,godot,AudioStream preview doesn't update after enabl...,### Tested versions\n\nReproducible in Godot v...,"bug,needs testing,topic:audio,topic:import",low,Minor


In [6]:
df["repo"].value_counts()

repo
pytorch                  14451
flutter                  13130
godot                    11207
rust                      9925
go                        9094
                         ...  
d3                           3
clean-code-javascript        2
fucking-algorithm            2
awesome                      1
awesome-go                   1
Name: count, Length: 68, dtype: int64

In [7]:
df = df.apply(lambda col: col.str.lower())

In [8]:
df['severity'].value_counts()

severity
critical    66491
minor       29769
major       17813
Name: count, dtype: int64

In [9]:
df["text"] = df["repo"].fillna(" ") + " " + df["title"].fillna(" ") + " " + df["body"].fillna(" ") + " " + df["labels"].fillna(" ") + " " + df["priority"].fillna(" ")
df.drop(['repo', 'title', 'body', 'labels', 'priority'], inplace=True, axis=1) 

In [12]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words='english', min_df=0.01, max_df=0.90)

text_vec = vectorizer.fit_transform(df['text'])

vectorizer.get_feature_names_out()

array(['00', '0000', '01', ..., 'zhaojuanmao', 'zip', 'zou3519'],
      shape=(1875,), dtype=object)

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(text_vec, df["severity"], test_size=0.2, random_state=42)

logreg = LogisticRegression(max_iter=100)

logreg.fit(X_train, y_train)

y_pred = logreg.predict(X_test)

print(accuracy_score(y_test, y_pred))

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names=logreg.classes_))


c:\Users\arsal\Desktop\support-ticket-triage-system\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.8319088319088319
              precision    recall  f1-score   support

    critical       0.95      0.94      0.95     13306
       major       0.59      0.38      0.46      3595
       minor       0.69      0.86      0.76      5914

    accuracy                           0.83     22815
   macro avg       0.74      0.73      0.73     22815
weighted avg       0.83      0.83      0.82     22815



In [ ]:
from sklearn.svm import LinearSVC

lsvc = LinearSVC()
lsvc.fit(X_train, y_train)
y_pred_lsvc = lsvc.predict(X_test)
print(accuracy_score(y_test, y_pred_lsvc))

0.8170501862809555---------------------------------0.7899627438088976

at min_df=0.2, max_df=0.9

0.7248301555993863--------------------------------0.719307473153627

at min_df=0.1, max_df=0.9

0.7557747096208635-------------------------------0.7493315801008109

at min_df=0.01, max_df=0.9

0.8319088319088319

              precision    recall  f1-score   support

    critical       0.95      0.94      0.95     13306
       major       0.59      0.38      0.46      3595
       minor       0.69      0.86      0.76      5914


    accuracy                           0.83     22815
    macro avg       0.74      0.73      0.73     22815
    weighted avg       0.83      0.83      0.82     22815